# Lesson 3.8d — Language Conditioning 的验证实验

模型**输入**了语言 ≠ 模型**使用**了语言 ≠ 模型**理解**了语言。本节把这三层变成可执行的判据。

三个测试从弱到强：**T1 correct**（正常语言）、**T2 shuffled**（随机打乱语言）、**T3 contradictory**（矛盾语言）。
另加一个 `drop` 模式，供 3.8.5 训练时做 instruction dropout。

对应 `docs/roadmap_v3.md` 的 3.8.4（后半）。3.8.5 的最小融合模型与 3.8.6 的消融接在本节之后。

## 运行说明

本 notebook 覆盖 **3.8.4.6 Language Conditioning 的验证实验**。

- 数据契约的实现在 `scripts/mml_contract.py`（**单一来源**）。下面这个 preamble cell 是**唯一**的前置：
  它把契约读进来并暴露 `datasets` / `episodes` / `pick` / `push` / `T_common` / `INSTRUCTIONS` / `VOCAB` /
  `LANGUAGE_IDS` / `build_sample` 等名字。3.8a-3.8d 都 import 同一份实现，所以字段布局与词表不会在
  几本 notebook 之间各自漂移——这是拆分之后最容易出的错。
- 执行顺序：`Kernel → Restart Kernel and Run All Cells`。
- 一个容易踩的 Jupyter 陷阱：**notebook 里显示的输出不一定属于当前 kernel。** 从磁盘重新加载
  notebook 时旧输出仍然显示，但 kernel 是空的——"看起来跑过了"和"状态还在"是两件事。

**命名约定**：`pick` / `push` 是**episode 列表**（`paired_episode_lists()` 给出）；
任务级字典写作 `datasets["PickCube-v1"]`。同一个名字不在同一本里兼指两件事。

In [1]:
# 前置：数据契约来自 scripts/mml_contract.py —— 单一实现。本 notebook 只 import，不重复声明。
import logging
import sys
import warnings
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import mml_contract as mmc

logging.getLogger("mani_skill").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*NVML.*")
warnings.filterwarnings("ignore", message=".*CUDA initialization.*")

datasets = mmc.load_datasets()
episodes = datasets["PickCube-v1"]["episodes"]
pick, push, T_common = mmc.paired_episode_lists(datasets)

INSTRUCTIONS, VOCAB, T_TXT, PAD = mmc.INSTRUCTIONS, mmc.VOCAB, mmc.T_TXT, mmc.PAD
LANGUAGE_IDS = mmc.LANGUAGE_IDS
ACTION_DIM, H = mmc.ACTION_DIM, mmc.H
IMAGE_HW, CHANNELS = mmc.IMAGE_HW, mmc.CHANNELS
PROPRIO_SLICES = datasets["PickCube-v1"]["proprio_slices"]
TASK_GOAL_SLICE = datasets["PickCube-v1"]["goal_slice"]
EXCLUDED_SLICES = datasets["PickCube-v1"]["excluded"]
OBS_DIM = datasets["PickCube-v1"]["state_dim"]
attrs = datasets["PickCube-v1"]["attrs"]
RGB_H5 = datasets["PickCube-v1"]["path"]

tokenize = mmc.tokenize
build_sample = mmc.build_sample
proprio_of = mmc.proprio_of
instruction_variants = mmc.instruction_variants

print(f"contract loaded from {Path(mmc.__file__).name}: {len(datasets)} tasks, "
      f"{sum(len(d['episodes']) for d in datasets.values())} episodes, T_common = {T_common}")

contract loaded from mml_contract.py: 2 tasks, 10 episodes, T_common = 50


## 3.8.4.6 Language Conditioning 的验证实验

这一节回答 VLA 研究里一个核心问题——**三个层次不是一回事**：

$$\underbrace{\text{输入了语言}}_{\text{the field exists}} \;\neq\; \underbrace{\text{使用了语言}}_{\text{the output depends on it}} \;\neq\; \underbrace{\text{理解了语言}}_{\text{generalises to unseen language}}$$

| 层次 | 可测的判据 | 本 pool 能达到吗 |
|---|---|---|
| **输入** | `language_ids` 存在、每任务不同、shape 正确 | ✅ 成立，但只有 **1 bit**（3.8.4.1：任务内 `H(ℓ) = 0`，跨任务 `H(ℓ) = 1.000` bit） |
| **使用** | **反事实**：换掉指令，动作要变，而且**方向正确** | ⚠️ 只能在**受限 probe** 上测（见下） |
| **理解** | 泛化到**没见过的指令**：新词、新组合、指称 | ❌ **不可测**：10 个词、2 条指令、单物体 |

### 三个测试，从弱到强

| # | 测试 | 变动什么 | 能证明什么 |
|---|---|---|---|
| **T1** | **Correct**（正常语言） | 什么都不变 | **最弱**：只能证明模型拟合了数据，**不能**证明它读了语言 |
| **T2** | **Shuffled**（随机打乱语言） | 同一条指令的**词序** | 见下面的架构预测——对我们是**阴性对照** |
| **T3** | **Contradictory**（矛盾语言） | **词袋**（另一条指令） | **最强**：state 帮不上忙时，只有指令能命名任务 |

**为什么 T2 与 T3 必须这样区分。** 只有 2 个任务时，"错的那条指令"就是"另一条"——如果把 T2 也理解成"给另一条指令"，**T2 与 T3 会坍缩成同一个测试**，三级阶梯就只剩两级。所以操作必须分开：

- **T2 = 词序**：`"pick up the cube"` → `"cube the up pick"`（**词袋不变**，只换顺序）
- **T3 = 词袋**：`"pick up the cube"` → `"push the cube to the goal"`（**词全换**）

### 一个可以在写模型之前就给出的架构预测

3.8.5 的最小融合模型会把语言 token 池化成一个向量。**任何均值/求和池化都是置换不变的**：

$$\operatorname{pool}\big(\text{permute}(z)\big) = \operatorname{pool}(z)$$

所以对本架构：**T2 造成的动作变化必须为 0（到浮点精度）**。这让 T2 从"语言测试"变成一个**实现的阴性对照**——如果 T2 让动作动了，要么语言编码器不是置换不变的，要么位置信息从别处漏了进来（比如加了 positional encoding）。**它检验实现是否诚实，不检验模型是否理解语言。**

**真正能测"使用"的只有 T3**，而 T3 在本 pool 上被 3.8.4.5 的泄漏地图挡住了（`image` 88.7%、`proprio` 从 `t=1` 起 98%、`task_goal` 10/10）。所以 T3 必须搬到一个**受限 probe** 上。

In [2]:
# 3.8.4.6：四个 instruction_mode。三个测试从弱到强，外加 3.8.5 训练时要用的 dropout 模式。

OTHER = {"PickCube-v1": "PushCube-v1", "PushCube-v1": "PickCube-v1"}


def shuffle_words(text, seed=0):
    """Permute the CONTENT words of one instruction: same bag of words, different order."""
    words = text.split()
    perm = np.random.default_rng(seed).permutation(len(words))
    if len(words) > 1 and np.array_equal(perm, np.arange(len(words))):
        perm = np.roll(perm, 1)                       # guarantee a real reordering
    return " ".join(words[i] for i in perm)


def instruction_variants(env_id):
    """The three tests of 3.8.4.6, plus the dropout mode 3.8.5 will train with."""
    correct_text = INSTRUCTIONS[env_id]
    shuffled_text = shuffle_words(correct_text, seed=0)
    assert shuffled_text != correct_text, "shuffle did not reorder anything"
    return {
        "correct":       tokenize(correct_text),                    # T1, weakest: fit only
        "shuffled":      tokenize(shuffled_text),                   # T2: word ORDER
        "contradictory": LANGUAGE_IDS[OTHER[env_id]].copy(),        # T3, strongest: the BAG
        "drop":          np.full(T_TXT, VOCAB["<pad>"], np.int64),  # 3.8.5's training dropout
    }


VARIANTS = {env_id: instruction_variants(env_id) for env_id in datasets}

INV_VOCAB = {v: k for k, v in VOCAB.items()}
print(f"{'task':>13} {'mode':>14}  {'text (content words)':<34} ids")
print("-" * 96)
for env_id in sorted(datasets):
    for mode, ids in VARIANTS[env_id].items():
        words = " ".join(INV_VOCAB[int(i)] for i in ids if int(i) > VOCAB["<eos>"])
        print(f"{env_id:>13} {mode:>14}  {words!r:<34} {ids.tolist()}")

# --- what makes the ladder a ladder, asserted --------------------------------
print()
for env_id in sorted(datasets):
    v = VARIANTS[env_id]
    for mode, ids in v.items():
        assert ids.shape == (T_TXT,) and ids.dtype == np.int64, (env_id, mode, ids.shape, ids.dtype)
    # T2 keeps the bag and changes only the order
    assert sorted(v["shuffled"].tolist()) == sorted(v["correct"].tolist()), "shuffled changed the bag"
    assert not np.array_equal(v["shuffled"], v["correct"]), "shuffled is identical to correct"
    # T3 changes the bag
    assert sorted(v["contradictory"].tolist()) != sorted(v["correct"].tolist()), \
        "contradictory kept the same bag of words"
    # drop carries nothing
    assert (v["drop"] == VOCAB["<pad>"]).all(), "drop is not all <pad>"
    # the two tasks are each other's contradictory instruction
    assert np.array_equal(VARIANTS[OTHER[env_id]]["contradictory"], v["correct"])
print("ladder asserted: T2 keeps the bag and changes the order, T3 changes the bag, drop carries")
print("nothing, and the two tasks are each other's contradictory instruction.")



         task           mode  text (content words)               ids
------------------------------------------------------------------------------------------------
  PickCube-v1        correct  'pick up the cube'                 [1, 3, 4, 5, 6, 2, 0, 0]
  PickCube-v1       shuffled  'the pick up cube'                 [1, 5, 3, 4, 6, 2, 0, 0]
  PickCube-v1  contradictory  'push the cube to the goal'        [1, 7, 5, 6, 8, 5, 9, 2]
  PickCube-v1           drop  ''                                 [0, 0, 0, 0, 0, 0, 0, 0]
  PushCube-v1        correct  'push the cube to the goal'        [1, 7, 5, 6, 8, 5, 9, 2]
  PushCube-v1       shuffled  'to cube goal the push the'        [1, 8, 6, 9, 5, 7, 5, 2]
  PushCube-v1  contradictory  'pick up the cube'                 [1, 3, 4, 5, 6, 2, 0, 0]
  PushCube-v1           drop  ''                                 [0, 0, 0, 0, 0, 0, 0, 0]

ladder asserted: T2 keeps the bag and changes the order, T3 changes the bag, drop carries
nothing, and the two ta

In [3]:
# --- the architectural prediction, COMPUTED rather than asserted --------------
EMB = np.random.default_rng(0).normal(size=(max(VOCAB.values()) + 1, 16))
PAD = VOCAB["<pad>"]


def language_embedding(ids):
    """Stand-in for 3.8.5's language encoder: look up tokens, mask padding, mean-pool.

    No positional term is added, which is the whole point of the T2 prediction below.
    """
    x = EMB[ids]
    keep = (ids != PAD).astype(np.float64)[:, None]
    return (x * keep).sum(0) / max(keep.sum(), 1.0)


print()
print("can the planned language encoder even see word order?")
print(f"  {'task':>13} {'|correct - shuffled|':>21} {'|correct - contradictory|':>27} {'|drop|':>10}")
for env_id in sorted(datasets):
    c = language_embedding(VARIANTS[env_id]["correct"])
    s = language_embedding(VARIANTS[env_id]["shuffled"])
    x = language_embedding(VARIANTS[env_id]["contradictory"])
    d = language_embedding(VARIANTS[env_id]["drop"])
    print(f"  {env_id:>13} {np.abs(c - s).max():>21.3e} {np.abs(c - x).max():>27.3e} "
          f"{np.abs(d).max():>10.3e}")
    # mathematically exact zero; numerically only floating-point summation order differs
    assert np.abs(c - s).max() < 1e-12, "mean pooling saw the word order"
    assert np.abs(c - x).max() > 1e-2, "the contradictory instruction produced the same embedding"
    assert np.abs(d).max() < 1e-15, "drop did not produce an all-pad (zero) embedding"
print()
print("=> T2 is EXACTLY zero up to floating-point summation order, so on this architecture a")
print("   shuffled instruction must leave the action unchanged. T2 is a NEGATIVE CONTROL on the")
print("   implementation, not a test of language use.")
print("=> T3 changes the embedding, so T3 is the only one of the three that can detect use.")


can the planned language encoder even see word order?
           task  |correct - shuffled|   |correct - contradictory|     |drop|
    PickCube-v1             2.220e-16                   6.087e-01  0.000e+00
    PushCube-v1             1.110e-16                   6.087e-01  0.000e+00

=> T2 is EXACTLY zero up to floating-point summation order, so on this architecture a
   shuffled instruction must leave the action unchanged. T2 is a NEGATIVE CONTROL on the
   implementation, not a test of language use.
=> T3 changes the embedding, so T3 is the only one of the three that can detect use.


### 判据：为什么不能看 loss

三个理由：

1. **T2/T3 都不改变 target**，所以 loss 必然升高——"loss 升高"不含任何信息；
2. **语言盲模型也能从 state 抄任务**（`image` 88.7%，`proprio` 从 `t=1` 起 98%），所以 loss 低也不代表读了语言；
3. 要测的是**模型输出对指令的敏感度**，不是 loss：

$$\operatorname{sens}(o, \mathrm{T}) = \big\|\pi(o, \ell_{\text{correct}}) - \pi(o, \ell_{\mathrm{T}})\big\|$$

### 判据表

`drop`（全 `<pad>`）是 3.8.5 **训练时**就要掺进去的 dropout 模式——**必须在训练时做**。否则测试时才把指令抹掉等于把模型推到分布外，得到的"零敏感度"是 OOD 伪影，不是"忽略了语言"。

| | **sens ≈ 0** | **sens > 0** |
|---|---|---|
| **loss 低** | 抄了近路（T3 也 ≈ 0 → 语言确实没被用） | ✅ **真的在条件化**（若方向也正确） |
| **loss 高** | 什么都没学会 | 对语言敏感，但学错了方向 |

**方向也必须对。** 光"动作变了"不够，还要变得对。在受限 probe 上：

$$\big[\pi(o, \ell_{\text{pick}}) - \pi(o, \ell_{\text{push}})\big]_{\text{arm 7 dims}} \approx 0, \qquad \big[\ \cdot\ \big]_{\text{gripper}} \approx +2.0$$

所以完整判据是三件事：**T3 的 sens ≠ 0**、**方向正确**、**T2 恰好为 0**。

In [4]:
# 3.8.4.6：为什么必须把 T3 搬到受限 probe 上，以及语言最多值多少
# (a) t=0 是唯一 state 无法命名任务的一帧
print("t=0: the only frame at which the state cannot name the task")
for i in range(min(len(pick), len(push))):
    a = np.concatenate([pick[i]["all_state"][0][lo:hi] for lo, hi in pick[i]["proprio_slices"]])
    b = np.concatenate([push[i]["all_state"][0][lo:hi] for lo, hi in push[i]["proprio_slices"]])
    assert np.array_equal(a, b), f"episode {i}: t=0 proprio differs across tasks"
print(f"  proprio identical across all {min(len(pick), len(push))} episode pairs, dim = {len(a)}")
a0 = np.stack([e["action"][0] for e in pick]).mean(0).astype(np.float64)
b0 = np.stack([e["action"][0] for e in push]).mean(0).astype(np.float64)
print(f"  a_0 pick = {np.round(a0, 4).tolist()}")
print(f"  a_0 push = {np.round(b0, 4).tolist()}")
print(f"  max |a_0 pick - a_0 push| over the 7 arm dims = {np.abs(a0[:7] - b0[:7]).max():.6f}")
print(f"  gripper difference                            = {a0[7] - b0[7]:+.4f}")
assert np.allclose(a0[:7], b0[:7], atol=1e-6), "the arm action already differs at t=0"
assert abs((a0[7] - b0[7]) - 2.0) < 1e-6, "the gripper difference is not 2.0"
print()
print("=> at t=0 the state is IDENTICAL and the required action differs in exactly ONE dimension.")
print("   That satisfies the 3.5 criterion exactly and is the only such frame in this pool.")
print("   The restricted probe is therefore: t=0, proprio only, no image, no task_goal.")

# (b) the ceiling for ANY task-conditioned model on this pool (no training needed)
print()
print("ceiling for any task-conditioned model here (computable without training)")
A = {env_id: np.stack([e["action"][:T_common] for e in datasets[env_id]["episodes"]])
     for env_id in datasets}
grand = np.concatenate([A[t] for t in A]).reshape(-1, ACTION_DIM).mean(0)
tmean = {t: A[t].reshape(-1, ACTION_DIM).mean(0) for t in A}
between = sum(float(np.mean([(tmean[t][j] - grand[j]) ** 2 for t in A])) for j in range(ACTION_DIM))
within = sum(float(np.mean([np.mean((A[t][:, :, j] - tmean[t][j]) ** 2) for t in A]))
             for j in range(ACTION_DIM))
g_between = float(np.mean([(tmean[t][7] - grand[7]) ** 2 for t in A]))
print(f"  L_blind (grand mean)    = {between + within:.5f}")
print(f"  L_aware (per-task mean) = {within:.5f}")
print(f"  headroom = between-task = {between:.5f}   ({100 * between / (between + within):.2f}% of total)")
print(f"  gripper share of it     = {100 * g_between / between:.2f}%")
print(f"  gripper mean: pick {tmean['PickCube-v1'][7]:+.4f}   push {tmean['PushCube-v1'][7]:+.4f}")
assert 99.0 < 100 * g_between / between < 100.0, "the headroom is no longer gripper-dominated"
print()
print("  => the whole task-conditioned signal is ONE dimension, and a language-blind model can")
print("     reach this ceiling too -- which is exactly why offline loss cannot be the criterion.")

t=0: the only frame at which the state cannot name the task
  proprio identical across all 5 episode pairs, dim = 25
  a_0 pick = [0.0192, 0.3954, -0.0103, -1.9569, 0.001, 2.332, 0.7953, 1.0]
  a_0 push = [0.0192, 0.3954, -0.0103, -1.9569, 0.001, 2.332, 0.7953, -1.0]
  max |a_0 pick - a_0 push| over the 7 arm dims = 0.000000
  gripper difference                            = +2.0000

=> at t=0 the state is IDENTICAL and the required action differs in exactly ONE dimension.
   That satisfies the 3.5 criterion exactly and is the only such frame in this pool.
   The restricted probe is therefore: t=0, proprio only, no image, no task_goal.

ceiling for any task-conditioned model here (computable without training)
  L_blind (grand mean)    = 1.02689
  L_aware (per-task mean) = 0.45271
  headroom = between-task = 0.57418   (55.91% of total)
  gripper share of it     = 99.54%
  gripper mean: pick +0.5120   push -1.0000

  => the whole task-conditioned signal is ONE dimension, and a language-bl

### 读法与边界

**三级阶梯在本 pool 上的实际可达性：**

| 层次 | 结论 |
|---|---|
| **输入**了语言 | ✅ 成立，但只是 **1 bit**（3.8.4.1：任务内 `H(ℓ) = 0`，跨任务 `1.000` bit） |
| **使用**了语言 | ⚠️ 主实验（臂 C）**预期为否定**；只有受限 probe（`t=0` + `proprio`-only）能干净地测 T3 |
| **理解**了语言 | ❌ 不可测——需要没见过的指令，本 pool 只有 10 个词、2 条指令、单物体 |

**边界：**

1. **受限 probe 是刻意构造的**：它拿掉了 `image`。所以它回答的是"**语言能否被用上**"，不是"语言在这个真实策略里是否被用了"。后者需要 **state 与画面都不可分**的任务对——**数据设计问题（P1）**。
2. **`t=0` 每个任务只有 5 个样本**（共 10 个）。10/10 vs 5/10 的二项检验 p ≈ 0.001，够用但很薄。
3. **probe 只测一个维度**：`t=0` 的动作差异全在 gripper。所以它测的是"**指令能否选对夹爪的符号**"——这是"使用语言"的**最小**实例，不是充分证据。
4. **closed loop 可以自我纠正**：`t≥1` 的 state 反馈能把 `t=0` 猜错的夹爪重开。所以 probe 上的成败**不能**直接翻译成 closed-loop success；终判仍是 P0 的 **closed-loop success**。
5. **T2 = 0 是架构的推论，不是数据的结论。** 换一个带 positional encoding 的语言编码器，T2 立刻变成有信息量的测试。所以报告 T2 读数时必须**绑定额定架构**。
6. **"打乱"在本 pool 里必须指词序。** 若指"换成另一条指令"，T2 与 T3 因只有 2 个任务而坍缩——这也是一个数据设计限制：**三级阶梯要到 3 条以上指令才真正成立**。

## 小结

1. **输入 / 使用 / 理解是三个层次**。契约里有 `language_ids` 只证明第一层，而且只有 **1 bit**。
2. **三级阶梯必须是三个不同的操作**：T1 correct（只证明拟合）、T2 shuffled（**词序**）、
   T3 contradictory（**词袋**）。两个任务下"错的那条指令"就是"另一条"，所以 T2 若按任务交换构造
   就会坍缩进 T3——**真正的三级阶梯需要 ≥3 条指令**。
3. **T2 在本架构上是阴性对照，不是语言测试**：均值池化置换不变，实测 `|correct − shuffled| = 2.2e-16`
   （只是浮点求和顺序），而 `|correct − contradictory| = 0.609`、`|drop|` 恰好为 0。
   **T2 动了就说明实现不诚实**（或位置信息漏了进来）。报告 T2 读数必须绑定额定架构。
4. **离线 loss 不能当判据**：T2/T3 都不改变 target，loss 必升，不含信息量；而语言盲模型本来就能从
   state 抄任务。要测的是 **instruction sensitivity**，判据是三件事：**T3 的 sens ≠ 0、方向正确、T2 == 0**。
5. **`drop` 必须在训练时做**（instruction dropout）。只在测试时抹掉指令得到的是 OOD 伪影，
   不是"忽略了语言"。
6. **受限 probe**：`t=0` 是唯一 state 无法命名任务的一帧——proprio 逐 episode 完全相同（差恰为 0），
   7 个 arm 动作完全相同，gripper 相差恰好 **+2.0**。这正是 3.5 判据被精确满足的地方。
   probe = `t=0` + 只用 `proprio` + 去掉 `image` 与 `task_goal`。
7. **上界（不需要训练）**：`L_blind = 1.02689`、`L_aware = 0.45271`、headroom **55.91%**，
   其中 gripper 占 **99.54%**。整个任务条件信号就是一个维度，而语言盲模型也能达到它。

## 自检

1. T1（correct）为什么"最弱"？它能证明什么、不能证明什么？
2. 为什么 T2 必须构造成**词序**置换，而不是"换成另一条指令"？按后者构造，三级阶梯会剩几级？
3. 如果语言编码器加了 positional encoding，T2 会变成什么样的测试？这说明"T2 = 0"是**数据**的性质
   还是**架构**的性质？
4. 为什么"shuffle 之后 loss 升高"不能作为模型使用了语言的证据？
5. 受限 probe 为什么必须去掉 `image`？去掉之后，这个实验还能回答原来的问题吗？
6. `t=0` 上 gripper 的动作差恰好是 `+2.0`。这个 2.0 从哪来？为什么它能当作"方向正确"的判据？